In [ ]:
import numpy as np
import torch
import torch.nn as nn

Modern BERT structure

- input embedding
=

In [ ]:
class FlashAttention(nn.Module):
  def __init__(self, d_model, d_k, d_v, num_head, block_size=128):
    super().__init__()
    self.d_k = d_k
    self.d_v = d_v
    self.num_head = num_head
    self.block_size = block_size

    self.W_q = nn.Linear(d_model, num_head * d_k, bias=False)
    self.W_k = nn.Linear(d_model, num_head * d_k, bias=False)
    self.W_v = nn.Linear(d_model, num_head * d_v, bias=False)
    self.W_o = nn.Linear(num_head * d_v, d_model, bias=False)

  def flash_attention(self, Q, K, V):
    """ Compute the flash attention between Q, K, V
    Arguments:
      Inputs:
        Q (N, num_head, seq_length, d_k)
        K (N, num_head, seq_length, d_k)
        V (N, num_head, seq_length, d_v)

      Output:
        attention_output: (N, num_head, seq_length, d_v)
    """
    N, num_head, seq_length, d_k = Q.shape
    block_size = min(self.block_size, seq_length)

    attention_output = torch.zeros_like(V)

    for i in range(0, seq_length, block_size):
      for j in range(0, seq_length, block_size):
        Q_block = Q[:, :, i:i+block_size, :] # (N, num_head, block_size, d_k)
        K_block = K[:, :, i:i+block_size, :] # (N, num_head, block_size, d_k)
        V_block = V[:, :, i:i+block_size, :] # (N, num_head, block_size, d_v)

        QK_T = torch.matmul(Q_block, K_block.transpose(-2, -1)) / np.sqrt(self.d_k)  # (N, num_head, block_size, block_size)

        # numerical stability softmax mentioned in the paper
        # computing softmax(QK.T)
        QK_T = QK_T - torch.max(QK_T, dim=-1, keepdim=True)[0]
        attn_block = torch.exp(QK_T)
        attn_block = attn_block / torch.sum(attn_block, dim=-1, keepdim=True)  # (N, num_head, block_size, block_size)

        # appply attention to V block
        attn_V = torch.matmul(attn_block, V_block) # (N, num_head, block_size, d_v)


        attention_output[:, :, i:i+block_size, :] += attn_V

    return attention_output


  def forward(self, Q, K, V):
    '''
    Inputs:
      Q, K, V: (N, seq_length, d_model)
    Output:
      attention (N, seq_length, d_model)
    '''
    N, seq_length, _ = Q.shape
    Q = self.W_q(Q).view(N, seq_length, self.num_head, self.d_k).transpose(1, 2)  # (N, num_head, seq_length, d_k)
    K = self.W_k(K).view(N, seq_length, self.num_head, self.d_k).transpose(1, 2)  # (N, num_head, seq_length, d_k)
    V = self.W_v(V).view(N, seq_length, self.num_head, self.d_v).transpose(1, 2)  # (N, num_head, seq_length, d_v)


    attention = self.flash_attention(Q, K, V) # (N, num_head, seq_length, d_v)

    attention = attention.transpose(1, 2).contiguous().view(N, seq_length, -1) # (N, seq_length, num_head * d_v)
    attention = self.W_o(attention)  # (N, seq_length, d_model)
    return attention

In [ ]:
class AttentionLayer(nn.Module):
  def init(self, layer_num, batch_size, seq_length, embedding_dim):
    super().__init__()
    self.layerNorm = nn.LayerNorm(torch.randn(batch, sentence_length, embedding_dim))
    self.ff = nn.ModuleList([nn.Linear(embedding_dim, 2*embedding_dim), nn.Linear(2*embedding_dim, embedding_dim)])
    self.mha = FlashAttention(embedding_dim, d_k, d_v, num_head, block_size=128)


  def forward(self, x):
    ''' AttentionLayer of ModernBERT
    - use pre normalization
    - layer_num % 3 == 0: flashAttention3 + global attention
    - lyaer_num % 3 != 0: flashAttention2 + local attention(128)
    - no activation except decoder
    - use GeGLU activation
    - RoPE

    arguments:
      x: Q=K=V (N, seq_length, embedding_dim)
    output:
      x: (N, seq_length, embedding_dim)
    '''
    # x = Q, K, V
    x_prime = self.layerNorm(x)
    x = self.mha(x_prime)
    x = x + x_prime

    x_prime = self.layerNorm(x)
    x = self.ff(x_prime)
    x = x + x_prime

    return x

In [ ]:
class ModernBERT(nn.Module):
  def init(self, num_layers):
    super().__init__()
    self.num_layers = num_layers
    self.input_embedding =
    self.attention_layers =
    self.decoder_layer =


  def forward(self, x):
    x = self.input_embedding(x)
    for i in range(num_layers):
      x = attention_layers[i](x)


    x = self.decoder_layer(x)
    return x